In [1]:
%matplotlib inline
import sys
import os
import time
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl

In [2]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import ticker
#from mpl_toolkits.basemap import Basemap, addcyclic, shiftgrid

# Model 1 trajectory

In [3]:
# Linear operator
def lin_op(C,
           g_1,
           g_1_tilde,
           g_2,
           g_2_tilde,
           b_1,
           b_2):
    return np.array(
        [
        [-C,0,g_1_tilde,0,0,0],
        [0,-C,b_1,0,0,0],
        [-g_1,-b_1,-C,0,0,0],
        [0,0,0,-C,0,g_2_tilde],
        [0,0,0,0,-C,b_2],
        [0,0,0,-g_2,-b_2,-C]
        ])

# Nonlinear operator
def nonlinear_op(x,a_1,a_2,d_1,d_2,e):
    return np.array([0,-a_1*x[0]*x[2]-d_1*x[3]*x[5],
                    +a_1*x[0]*x[1]+d_1*x[3]*x[4],
                    e*x[1]*x[5]-e*x[2]*x[4],
                    -a_2*x[0]*x[5]-d_2*x[3]*x[2],
                    a_2*x[0]*x[4]+d_2*x[3]*x[1]])

# Variance of the Wiener process
def Sigma(s):
    return np.array([s, s, s, s, s, s])

# Forcing
def forcing(C,x_1_star,x_4_star):
    return np.array([C*x_1_star,0,0,C*x_4_star,0,0])

In [4]:
# parameters
def a_m(m,b): return (8*np.sqrt(2)/np.pi) * ((m**2)/(4*m**2 - 1)) * ((b**2 + m**2 -1)/(b**2 + m**2))

def b_m(m,b,beta): return (beta*b**2)/(b**2 + m**2)

def d_m(m,b): return (64*np.sqrt(2)/(15*np.pi)) * ((b**2 - m**2 +1)/(b**2 + m**2))
    
def g_m_tilde(m,b,gamma): return gamma * ((4*m)/(4*m**2 -1)) * (b*np.sqrt(2)/np.pi)

e = 16*np.sqrt(2)/(5*np.pi)

def g_m(m,b,gamma): return gamma * ((4*m**3)/(4*m**2 -1)) * (b*np.sqrt(2)/(np.pi*(b**2 + m**2)))

In [5]:
def CdV_model(x_0,F,L,S,a_1,a_2,d_1,d_2,e,dt,N,rand_seed = 123):
    
    #np.random.seed(rand_seed)
    
    # Create a local random generator (this way I do not affect the random seed globally)
    rng = np.random.default_rng(rand_seed)
    
    ### Inputs
    # x_0: initial condition. E.g. np.array([0.01,0.2,0.3])
    # F: forcing
    # L: linear operator.
    # B: nonlinear operator
    # S: variance of the Wiener process
    # dt: integration time stepping
    # N: number of integration steps
    
    # Array for results
    x = np.zeros([N+1,len(x_0)])
    
    # Set initial condition
    x[0] = x_0
    
    # Defined Wiener process
    wiener = rng.normal(0, 1, (N,len(x_0)))*np.sqrt(dt)
    
    # forward integration via Euler-Mayurama scheme
    for t in range(N):
        # Deterministic part
        det_part = (F + np.matmul(L,x[t]) + nonlinear_op(x[t],a_1,a_2,d_1,d_2,e))*dt
        # Stochastic part
        stoch_part = S*wiener[t]
        x[t+1] = x[t] + det_part + stoch_part

    return x

# Parameters

In [6]:
x_1_star = 0.95
x_4_star = -0.76095
C = 0.1
gamma = 0.2
beta = 1.25
b = 1.6
s = 0.05

# What worked well was dt = 0.01, s = 0.05, b = 1.6. However, it looks like we should have dt = 0.001 
# for a more precise scheme

a_1, a_2, b_1, b_2 = a_m(1,b), a_m(2,b), b_m(1,b,beta), b_m(2,b,beta)
d_1, d_2 = d_m(1,b), d_m(2,b)
g_1_tilde, g_2_tilde = g_m_tilde(1,b,gamma), g_m_tilde(2,b,gamma)
g_1, g_2 = g_m(1,b,gamma), g_m(2,b,gamma)

In [7]:
L = lin_op(C,g_1,g_1_tilde,g_2,g_2_tilde,b_1,b_2)
F = forcing(C,x_1_star,x_4_star)
S = Sigma(s)

In [17]:
# A dt = 0.01 and s = 0.02 is good

## Simulate a long trajectory

In [8]:
dt = 0.01
N = 10000000 + 100000
x_0 = np.random.normal(0, 1, 6) # We start from a random initial condition and then remove a transient after

# Generate a random seed (or set it manually for reproducibility)
rand_seed = np.random.randint(0, 2**32 - 1)  # Random seed
orbit = CdV_model(x_0,F,L,S,a_1,a_2,d_1,d_2,e,dt,N,rand_seed = rand_seed)[100000:]

In [9]:
np.save('./results/CdV_long_trajectory.npy',orbit)

### ---> Mean and variance for the invariant density

In [13]:
trajectory = np.load('./results/CdV_long_trajectory.npy')
mean = np.mean(trajectory,0)
sigmas = np.std(trajectory,0)
np.save('./results/mean_invariant_density.npy',mean)
np.save('./results/sigma_invariant_density.npy',sigmas)

In [14]:
mean

array([ 0.71046255, -0.14506166, -0.12400285, -0.38420025,  0.06674518,
        0.06680191])

In [15]:
sigmas

array([0.2183917 , 0.26545972, 0.24538472, 0.16845988, 0.14155081,
       0.13319658])